In [1]:
from csrio_image2biomass.configs.settings import AUGUMENTED_DATA_DIR
import polars as pl
test = pl.read_csv(AUGUMENTED_DATA_DIR / "test.csv")
test.sort("image_path").head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i64,i64,i64,i64,i64
"""test/ID1001187975.jpg""",0,0,0,0,0


In [2]:
from csrio_image2biomass.utils.dataset import BiomassDataset
from torch.utils.data import DataLoader
    
test_dataset = BiomassDataset(dataframe=test, img_dir=AUGUMENTED_DATA_DIR)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=12)

for batch in test_loader:
    images, labels = batch
    print(f"Image batch shape: {images.size()}")
    print(f"Label batch shape: {labels.size()}")
    break


Image batch shape: torch.Size([1, 3, 448, 224])
Label batch shape: torch.Size([1, 5])


In [3]:
import torch
import torchvision.models as models
import torch.nn as nn

class BiomassModel(nn.Module):
    def __init__(self):
        super(BiomassModel, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.fc = nn.Linear(self.backbone.fc.out_features, 5)

    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return torch.relu(x)
    
model = BiomassModel()
model.load_state_dict(torch.load('biomass_model.pth', map_location=torch.device('cpu')))

<All keys matched successfully>

In [4]:
model.eval()  # Set to evaluation mode

outputs = []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

array([[11.062555  , 24.751425  ,  0.12489364, 12.967254  , 37.139153  ]],
      dtype=float32)

In [5]:
output_df = pl.DataFrame(outputs, schema=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"])
output_df

Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
f32,f32,f32,f32,f32
11.062555,24.751425,0.124894,12.967254,37.139153


In [6]:
test_df = test.select("image_path").hstack(output_df).unpivot(on=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"], index="image_path", value_name="target")
test_df

image_path,variable,target
str,str,f32
"""test/ID1001187975.jpg""","""Dry_Clover_g""",11.062555
"""test/ID1001187975.jpg""","""Dry_Dead_g""",24.751425
"""test/ID1001187975.jpg""","""Dry_Green_g""",0.124894
"""test/ID1001187975.jpg""","""Dry_Total_g""",12.967254
"""test/ID1001187975.jpg""","""GDM_g""",37.139153


In [7]:
# concat image_path and variable columns
submission = test_df.with_columns(
    pl.concat_str([pl.col("image_path").str.split('/').list.get(-1).str.split('.jpg').list.get(0), pl.lit("__"), pl.col("variable")]).alias("sample_id")
).select(["sample_id", "target"])
print(submission)

shape: (5, 2)
┌────────────────────────────┬───────────┐
│ sample_id                  ┆ target    │
│ ---                        ┆ ---       │
│ str                        ┆ f32       │
╞════════════════════════════╪═══════════╡
│ ID1001187975__Dry_Clover_g ┆ 11.062555 │
│ ID1001187975__Dry_Dead_g   ┆ 24.751425 │
│ ID1001187975__Dry_Green_g  ┆ 0.124894  │
│ ID1001187975__Dry_Total_g  ┆ 12.967254 │
│ ID1001187975__GDM_g        ┆ 37.139153 │
└────────────────────────────┴───────────┘


In [8]:
submission.write_csv("submission.csv")